# Experiment 04 — Perch Probe Tuning

**Builds on:** Exp02's deduplicated Perch v2 soundscape-window baseline.

**New in this experiment:** sweep per-class logistic-regression regularization, optional class balancing, and the blend weight between tuned probes and raw Perch scores.

**Hypothesis:** The current Perch LR probe may be overfitting the small labeled soundscape set. A more conservative probe and partial blend with raw Perch probabilities should improve padded cmAP without expensive model training.

In [1]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.special import expit as sigmoid
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# Run correctly from either the project root or the notebooks directory.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
CACHE_DIR = DATA_DIR / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

EMB_CACHE = CACHE_DIR / "exp02_embeddings.npy"
LOG_CACHE = CACHE_DIR / "exp02_logits.npy"
RESULTS_CSV = CACHE_DIR / "exp04_probe_tuning_results.csv"
RESULTS_JSON = CACHE_DIR / "exp04_probe_tuning_best.json"

N_SPLITS = 5
PCA_DIM = 64
MIN_POS = 3
C_GRID = [0.05, 0.10, 0.25, 0.50, 1.00]
CLASS_WEIGHT_GRID = [None, "balanced"]
BLEND_GRID = [0.25, 0.50, 0.75, 1.00]

print("Project root:", PROJECT_ROOT)
print("Embedding cache exists:", EMB_CACHE.exists())
print("Logit cache exists:", LOG_CACHE.exists())

Project root: /Users/jroessler/Development/kaggle/birdclef
Embedding cache exists: True
Logit cache exists: True


In [2]:
# Load taxonomy, labels, and cached Perch outputs from exp02.
taxonomy = pd.read_csv(DATA_DIR / "taxonomy.csv")
N_CLASSES = len(taxonomy)
taxon_ids = taxonomy["primary_label"].astype(str).tolist()
taxon_to_idx = {tid: i for i, tid in enumerate(taxon_ids)}

perch_labels = pd.read_csv(DATA_DIR / "models/perch_tf/assets/labels.csv", header=0)
perch_labels.columns = ["scientific_name"]
perch_labels["bc_index"] = np.arange(len(perch_labels))

mapping = taxonomy.merge(perch_labels, on="scientific_name", how="left")
mapped_mask = mapping["bc_index"].notna()
comp_to_bc = {
    taxon_to_idx[str(row["primary_label"])] : int(row["bc_index"])
    for _, row in mapping[mapped_mask].iterrows()
}

def time_to_sec(t):
    h, m, s = map(int, t.split(":"))
    return h * 3600 + m * 60 + s

raw_df = pd.read_csv(DATA_DIR / "train_soundscapes_labels.csv")
raw_df["start_sec"] = raw_df["start"].apply(time_to_sec)
labels_df = (
    raw_df
    .drop_duplicates(subset=["filename", "start_sec"])
    .sort_values(["filename", "start_sec"])
    .reset_index(drop=True)
)

def parse_labels(label_str):
    vec = np.zeros(N_CLASSES, dtype=np.float32)
    for tid in str(label_str).split(";"):
        tid = tid.strip()
        if tid in taxon_to_idx:
            vec[taxon_to_idx[tid]] = 1.0
    return vec

Y = np.stack(labels_df["primary_label"].apply(parse_labels).values)
emb_full = np.load(EMB_CACHE).astype(np.float32)
logits_full = np.load(LOG_CACHE).astype(np.float32)

scores_raw = np.zeros((len(labels_df), N_CLASSES), dtype=np.float32)
for comp_idx, bc_idx in comp_to_bc.items():
    scores_raw[:, comp_idx] = logits_full[:, bc_idx]
raw_probs = sigmoid(scores_raw)

groups = labels_df["filename"].values
folds = list(GroupKFold(n_splits=N_SPLITS).split(emb_full, groups=groups))

print(f"Mapped species: {mapped_mask.sum()}/{N_CLASSES}")
print(f"Labels: {Y.shape}, files: {labels_df['filename'].nunique()}, active classes: {(Y.sum(axis=0) > 0).sum()}")
print(f"Embeddings: {emb_full.shape}, logits: {logits_full.shape}")

Mapped species: 203/234
Labels: (739, 234), files: 66, active classes: 75
Embeddings: (739, 1536), logits: (739, 14795)


In [3]:
def padded_cmap(y_true, y_pred, pad=5):
    aps = []
    for c in range(y_true.shape[1]):
        yt = np.concatenate([y_true[:, c], np.ones(pad)])
        yp = np.concatenate([y_pred[:, c], np.ones(pad)])
        aps.append(average_precision_score(yt, yp))
    return float(np.mean(aps))

def macro_auc(y_true, y_pred):
    aucs = []
    for c in range(y_true.shape[1]):
        positives = y_true[:, c].sum()
        if 0 < positives < len(y_true):
            aucs.append(roc_auc_score(y_true[:, c], y_pred[:, c]))
    return float(np.mean(aucs)), len(aucs)

# Precompute fold-specific PCA features once so the sweep only refits cheap probes.
fold_features = []
for fold, (tr, va) in enumerate(folds):
    scaler = StandardScaler()
    pca = PCA(n_components=PCA_DIM, whiten=True, random_state=42)
    emb_tr = pca.fit_transform(scaler.fit_transform(emb_full[tr]))
    emb_va = pca.transform(scaler.transform(emb_full[va]))
    X_tr = np.concatenate([emb_tr, scores_raw[tr]], axis=1)
    X_va = np.concatenate([emb_va, scores_raw[va]], axis=1)
    active = [
        c for c in range(N_CLASSES)
        if MIN_POS <= Y[tr, c].sum() < len(tr)
    ]
    fold_features.append((tr, va, X_tr, X_va, active))
    print(f"Fold {fold}: train={len(tr)}, val={len(va)}, probes={len(active)}")

raw_auc, raw_auc_classes = macro_auc(Y, raw_probs)
raw_cmap = padded_cmap(Y, raw_probs)
print(f"Raw Perch mapped scores: AUC={raw_auc:.4f} ({raw_auc_classes} classes), cmAP={raw_cmap:.4f}")

Fold 0: train=590, val=149, probes=61


Fold 1: train=592, val=147, probes=60
Fold 2: train=591, val=148, probes=62
Fold 3: train=593, val=146, probes=54


Fold 4: train=590, val=149, probes=58


Raw Perch mapped scores: AUC=0.7367 (75 classes), cmAP=0.8617


In [4]:
rows = [{
    "method": "raw_perch",
    "C": np.nan,
    "class_weight": "none",
    "blend_probe": 0.0,
    "macro_auc": raw_auc,
    "auc_classes": raw_auc_classes,
    "padded_cmap": raw_cmap,
}]

for class_weight in CLASS_WEIGHT_GRID:
    cw_name = "none" if class_weight is None else class_weight
    for C in C_GRID:
        oof_probe = raw_probs.copy()
        fitted = 0

        for tr, va, X_tr, X_va, active in fold_features:
            for c in active:
                clf = LogisticRegression(
                    C=C,
                    class_weight=class_weight,
                    max_iter=1000,
                    solver="lbfgs",
                )
                clf.fit(X_tr, Y[tr, c])
                oof_probe[va, c] = clf.predict_proba(X_va)[:, 1]
                fitted += 1

        for alpha in BLEND_GRID:
            blend = (alpha * oof_probe) + ((1.0 - alpha) * raw_probs)
            auc, n_auc = macro_auc(Y, blend)
            cmap = padded_cmap(Y, blend)
            rows.append({
                "method": "perch_lr_probe_sweep",
                "C": C,
                "class_weight": cw_name,
                "blend_probe": alpha,
                "macro_auc": auc,
                "auc_classes": n_auc,
                "padded_cmap": cmap,
                "fitted_probes": fitted,
            })
        print(f"Finished class_weight={cw_name:<8} C={C:<4} fitted_probes={fitted}")

results = pd.DataFrame(rows).sort_values(["padded_cmap", "macro_auc"], ascending=False).reset_index(drop=True)
results.to_csv(RESULTS_CSV, index=False)

best = results.iloc[0].to_dict()
with open(RESULTS_JSON, "w") as f:
    json.dump(best, f, indent=2)

print("\nTop configurations")
print(results.head(12).to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("\nSaved:", RESULTS_CSV)
print("Saved:", RESULTS_JSON)

Finished class_weight=none     C=0.05 fitted_probes=295


Finished class_weight=none     C=0.1  fitted_probes=295


Finished class_weight=none     C=0.25 fitted_probes=295


Finished class_weight=none     C=0.5  fitted_probes=295


Finished class_weight=none     C=1.0  fitted_probes=295


Finished class_weight=balanced C=0.05 fitted_probes=295


Finished class_weight=balanced C=0.1  fitted_probes=295


Finished class_weight=balanced C=0.25 fitted_probes=295


Finished class_weight=balanced C=0.5  fitted_probes=295


Finished class_weight=balanced C=1.0  fitted_probes=295

Top configurations
              method      C class_weight  blend_probe  macro_auc  auc_classes  padded_cmap  fitted_probes
perch_lr_probe_sweep 0.0500     balanced       0.2500     0.8934           75       0.8993       295.0000
perch_lr_probe_sweep 0.1000     balanced       0.2500     0.8920           75       0.8992       295.0000
perch_lr_probe_sweep 0.2500     balanced       0.2500     0.8903           75       0.8989       295.0000
perch_lr_probe_sweep 0.0500     balanced       0.5000     0.8952           75       0.8989       295.0000
perch_lr_probe_sweep 0.1000     balanced       0.5000     0.8938           75       0.8988       295.0000
perch_lr_probe_sweep 0.2500         none       0.2500     0.8906           75       0.8987       295.0000
perch_lr_probe_sweep 0.1000         none       0.2500     0.8917           75       0.8986       295.0000
perch_lr_probe_sweep 0.5000     balanced       0.2500     0.8913           7

## Result Notes

Record the best row in `experiments.md` after execution. The baseline to beat is the deduplicated `exp02` Perch LR probe at `0.8960` macro AUC and `0.8974` padded cmAP.